In [1]:
!pip -q install pytest pytest-cov

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
%%writefile estructuras.py

from __future__ import annotations

from collections import Counter
from typing import Any, Dict, Iterable, List, Sequence, Tuple


def filtra_pares(nums: Iterable[int]) -> List[int]:
    """Devuelve una lista con los números pares, manteniendo el orden."""
    return [n for n in nums if n % 2 == 0]


def aplana(lista_de_listas: Iterable[Iterable[Any]]) -> List[Any]:
    """Aplana una lista de listas (n niveles)."""
    out: List[Any] = []
    for sub in lista_de_listas:
        out.extend(sub)
    return out


def aplana_x_dimesiones(lista_de_listas: Iterable[Iterable[Any]]) -> List[Any]:
    out: List[Any] = []
    for sub in lista_de_listas:
        aplana(sub)
    return out


def esta_ordenada(nums: Sequence[int], *, asc: bool = True) -> bool:
    """True si la secuencia está ordenada ascendente (o descendente si asc=False)."""
    if asc:
        return all(nums[i] <= nums[i + 1] for i in range(len(nums) - 1))
    return all(nums[i] >= nums[i + 1] for i in range(len(nums) - 1))


def sin_duplicados_preservando_orden(items: Iterable[Any]) -> List[Any]:
    """Elimina duplicados preservando el orden de aparición."""
    vistos = set()
    out = []
    for x in items:
        if x not in vistos:
            vistos.add(x)
            out.append(x)
    return out


def cuenta_frecuencias(items: Iterable[Any]) -> Dict[Any, int]:
    """Cuenta ocurrencias y devuelve dict item->frecuencia."""
    return dict(Counter(items))


def mezcla_dicts(base: Dict[str, Any], extra: Dict[str, Any], *, overwrite: bool = True) -> Dict[str, Any]:
    """
    Mezcla dos dicts. Si overwrite=False, no pisa claves ya existentes en base.
    Devuelve un nuevo dict (no muta los originales).
    """
    out = dict(base)
    for k, v in extra.items():
        if overwrite or k not in out:
            out[k] = v
    return out


def get_path(d: Dict[str, Any], path: str, *, sep: str = ".", default: Any = None) -> Any:
    """
    Accede a un diccionario anidado con una ruta tipo "a.b.c".
    Si falta alguna clave, devuelve default.
    """
    cur: Any = d
    for key in path.split(sep):
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def agrupa_por_clave(rows: Iterable[Dict[str, Any]], key: str) -> Dict[Any, List[Dict[str, Any]]]:
    """
    Agrupa una lista de dicts por una clave.
    Ej: [{"id":1,"g":"A"},{"id":2,"g":"A"}], key="g" -> {"A":[...,...]}
    """
    out: Dict[Any, List[Dict[str, Any]]] = {}
    for r in rows:
        if key not in r:
            raise KeyError(f"Falta la clave '{key}' en {r}")
        out.setdefault(r[key], []).append(r)
    return out


def valida_schema_minimo(obj: Dict[str, Any], required: Sequence[str]) -> Tuple[bool, List[str]]:
    """
    Valida que existan ciertas claves. Devuelve (ok, faltantes).
    """
    faltan = [k for k in required if k not in obj]
    return (len(faltan) == 0, faltan)


# ---- Opcional: NumPy (si está instalado) ----

def normaliza_vector(v: Sequence[float]) -> List[float]:
    """
    Normaliza un vector para que sume 1.
    Lanza ValueError si la suma es 0.
    """
    total = float(sum(v))
    if total == 0.0:
        raise ValueError("No se puede normalizar un vector con suma 0")
    return [float(x) / total for x in v]


def dot(a: Sequence[float], b: Sequence[float]) -> float:
    """Producto escalar simple (sin numpy)."""
    if len(a) != len(b):
        raise ValueError("Vectores de distinta longitud")
    return float(sum(x * y for x, y in zip(a, b)))

Writing estructuras.py


In [2]:
%%writefile test_estructuras.py

import pytest

from estructuras import (
    agrupa_por_clave,
    aplana,
    cuenta_frecuencias,
    dot,
    esta_ordenada,
    filtra_pares,
    get_path,
    mezcla_dicts,
    normaliza_vector,
    sin_duplicados_preservando_orden,
    valida_schema_minimo,
)


def test_filtra_pares():
    assert filtra_pares([1, 2, 3, 4, 5, 6]) == [2, 4, 6]
    assert filtra_pares([]) == []
    assert filtra_pares([1, 3, 5]) == []


def test_aplana():
    assert aplana([[1, 2], [3], [], [4, 5]]) == [1, 2, 3, 4, 5]


@pytest.mark.parametrize("nums, asc, esperado", [
    ([1, 2, 2, 5], True, True),
    ([5, 2, 2, 1], True, False),
    ([5, 2, 2, 1], False, True),
    ([], True, True),
    ([7], True, True),
])
def test_esta_ordenada(nums, asc, esperado):
    assert esta_ordenada(nums, asc=asc) is esperado


def test_sin_duplicados_preservando_orden():
    assert sin_duplicados_preservando_orden([1, 2, 2, 3, 1]) == [1, 2, 3]
    assert sin_duplicados_preservando_orden(["a", "a", "b", "a"]) == ["a", "b"]


def test_cuenta_frecuencias():
    assert cuenta_frecuencias(["a", "b", "a"]) == {"a": 2, "b": 1}
    assert cuenta_frecuencias([]) == {}


def test_mezcla_dicts_overwrite_true():
    base = {"a": 1, "b": 2}
    extra = {"b": 99, "c": 3}
    assert mezcla_dicts(base, extra, overwrite=True) == {"a": 1, "b": 99, "c": 3}
    # no muta originales
    assert base == {"a": 1, "b": 2}
    assert extra == {"b": 99, "c": 3}


def test_mezcla_dicts_overwrite_false():
    base = {"a": 1, "b": 2}
    extra = {"b": 99, "c": 3}
    assert mezcla_dicts(base, extra, overwrite=False) == {"a": 1, "b": 2, "c": 3}


def test_get_path():
    d = {"a": {"b": {"c": 10}}}
    assert get_path(d, "a.b.c") == 10
    assert get_path(d, "a.b.x") is None
    assert get_path(d, "a.b.x", default=0) == 0


def test_agrupa_por_clave():
    rows = [
        {"id": 1, "g": "A"},
        {"id": 2, "g": "A"},
        {"id": 3, "g": "B"},
    ]
    grouped = agrupa_por_clave(rows, "g")
    assert set(grouped.keys()) == {"A", "B"}
    assert [r["id"] for r in grouped["A"]] == [1, 2]
    assert [r["id"] for r in grouped["B"]] == [3]


def test_agrupa_por_clave_falta_key():
    with pytest.raises(KeyError, match="Falta la clave"):
        agrupa_por_clave([{"id": 1}], "g")


def test_valida_schema_minimo():
    ok, faltan = valida_schema_minimo({"a": 1, "b": 2}, required=["a", "b", "c"])
    assert ok is False
    assert faltan == ["c"]

    ok, faltan = valida_schema_minimo({"a": 1, "b": 2, "c": 3}, required=["a", "c"])
    assert ok is True
    assert faltan == []


def test_normaliza_vector():
    assert normaliza_vector([1, 1, 2]) == pytest.approx([0.25, 0.25, 0.5])


def test_normaliza_vector_error():
    with pytest.raises(ValueError, match="suma 0"):
        normaliza_vector([0, 0, 0])


def test_dot():
    assert dot([1, 2, 3], [4, 5, 6]) == 32.0  # 1*4 + 2*5 + 3*6
    with pytest.raises(ValueError):
        dot([1, 2], [1])

Writing test_estructuras.py


In [4]:
!pytest -q

..................                                                       [100%]
18 passed in 0.05s
